# Contextual Cost + Review LogEI Campaign

This notebook demonstrates the v2.3 contextual cost-review workflow using YAML config and a copied CSV working log.

In [ ]:
from pathlib import Path
import math
import shutil

from bo_forge import CampaignSession

PROJECT_ROOT = Path.cwd().resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "20_contextual_cost_review_logei.yaml"
SEED_LOG_PATH = PROJECT_ROOT / "examples" / "20_contextual_cost_review_campaign_log.csv"
WORKING_LOG_PATH = PROJECT_ROOT / "examples" / "20_contextual_cost_review_working_log.csv"
LATEST_SUGGESTIONS_PATH = PROJECT_ROOT / "examples" / "20_contextual_cost_review_latest_suggestions.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

TARGET_OBSERVED_ROWS = 15

shutil.copyfile(SEED_LOG_PATH, WORKING_LOG_PATH)


Load the campaign from the committed config and the copied working log. The committed seed log remains unchanged.

In [ ]:
campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)
campaign.validate()
campaign.summary()


In [ ]:
campaign.context_summary(), campaign.cost_summary(), campaign.next_action()


Generate one dry-run suggestion at a fixed context. Because review is enabled, the row is appended as pending review and must be accepted explicitly before observation.

In [ ]:
context_values = {"feedstock_acidity": 0.50}
suggestions = campaign.suggest_next(context_values=context_values)
suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
suggestions


The objective and actual cost below are deterministic tutorial functions, not part of BO Forge.

In [ ]:
def simulate_yield(row):
    loading = float(row["catalyst_loading"])
    temperature = float(row["reaction_temperature"])
    acidity = float(row["feedstock_acidity"])
    solvent_bonus = {"MeCN": 0.08, "EtOH": 0.03, "Water": -0.05}[str(row["solvent"])]
    loading_term = 0.95 + 0.72 * loading - 1.05 * (loading - 0.55) ** 2
    temperature_term = -0.00008 * (temperature - 94.0) ** 2
    context_term = 0.24 * acidity - 0.30 * (acidity - 0.55) ** 2
    smooth_variation = 0.02 * math.sin(6.0 * loading + 0.03 * temperature + 2.5 * acidity)
    return round(loading_term + temperature_term + context_term + solvent_bonus + smooth_variation, 6)


def simulate_actual_cost(row):
    estimate = float(row["cost_estimate"])
    loading = float(row["catalyst_loading"])
    return round(estimate * (1.0 + 0.02 * math.sin(5.0 * loading)), 6)


In [ ]:
campaign.append_suggestions(suggestions)
for row_id in suggestions["row_id"]:
    campaign.review_suggestion(row_id, "accepted", note="tutorial approval")
    row = campaign.df.loc[campaign.df["row_id"] == row_id].iloc[0]
    campaign.mark_observed(
        row_id,
        objective_value=simulate_yield(row),
        actual_cost=simulate_actual_cost(row),
    )

campaign.summary()


Run a small sequential loop across a few context values. Suggestions remain non-mutating until appended, review decisions are explicit, and actual cost is recorded when marking observations.

In [ ]:
context_cycle = [0.25, 0.50, 0.75]
records = []
while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    context_values = {"feedstock_acidity": context_cycle[len(records) % len(context_cycle)]}
    suggestions = campaign.suggest_next(context_values=context_values)
    suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
    campaign.append_suggestions(suggestions)
    for row_id in suggestions["row_id"]:
        campaign.review_suggestion(row_id, "accepted", note="tutorial approval")
        row = campaign.df.loc[campaign.df["row_id"] == row_id].iloc[0]
        objective_value = simulate_yield(row)
        actual_cost = simulate_actual_cost(row)
        campaign.mark_observed(row_id, objective_value=objective_value, actual_cost=actual_cost)
        records.append(
            {
                "row_id": row_id,
                "feedstock_acidity": row["feedstock_acidity"],
                "yield_score": objective_value,
                "actual_cost": actual_cost,
            }
        )

len(campaign.observed_data()), records[-3:]


In [ ]:
campaign.context_summary(), campaign.cost_summary()


In [ ]:
report_path = REPORTS_DIR / "20_contextual_cost_review_report.md"
context_plot_path = REPORTS_DIR / "20_contextual_cost_review_context_diagnostics.png"
cost_plot_path = REPORTS_DIR / "20_contextual_cost_review_cost_progress.png"
progress_plot_path = REPORTS_DIR / "20_contextual_cost_review_progress.png"

action_report = campaign.export_report(report_path)
campaign.plot_context_diagnostics(save_path=context_plot_path)
campaign.plot_cost_progress(save_path=cost_plot_path)
campaign.plot_progress(save_path=progress_plot_path)

action_report, context_plot_path, cost_plot_path, progress_plot_path


CLI equivalents:

```bash
python -m bo_forge validate --config configs/20_contextual_cost_review_logei.yaml --log examples/20_contextual_cost_review_campaign_log.csv
python -m bo_forge suggest --config configs/20_contextual_cost_review_logei.yaml --log examples/20_contextual_cost_review_campaign_log.csv --context feedstock_acidity=0.5
python -m bo_forge context-summary --config configs/20_contextual_cost_review_logei.yaml --log examples/20_contextual_cost_review_campaign_log.csv
python -m bo_forge cost-summary --config configs/20_contextual_cost_review_logei.yaml --log examples/20_contextual_cost_review_campaign_log.csv
```

Limitations: contextual qLogNEI, contextual qLogNEHVI, contextual multi-objective, contextual structured, contextual multi-fidelity, and contextual replicate workflows remain deferred.